In [1]:
!pip install kaggle
!kaggle datasets download -d gqfiddler/scotus-opinions

Dataset URL: https://www.kaggle.com/datasets/gqfiddler/scotus-opinions
License(s): CC-BY-NC-SA-4.0
100% 233M/233M [00:02<00:00, 112MB/s]



In [2]:
!unzip scotus-opinions.zip -d data/raw/

Archive:  scotus-opinions.zip
  inflating: data/raw/all_opinions.csv  
  inflating: data/raw/cleaning_functions.py  
  inflating: data/raw/opinion loading and parsing_2020.ipynb  
  inflating: data/raw/opinions_since_1970.csv  


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/all_opinions.csv")
texts_raw = df["text"].dropna().tolist()  # column name may vary — check with df.columns

In [4]:
print(type(texts_raw))
print(len(texts_raw))
print(texts_raw[0][:300])

<class 'list'>
35781
There is no right more basic in our democracy than the
right to participate in electing our political leaders. Citi-
zens can exercise that right in a variety of ways: They can
run for office themselves, vote, urge others to vote for a
particular candidate, volunteer to work on a campaign,
and contr


In [5]:
import os
import re
import time
from collections import Counter
from datetime import datetime
from getpass import getpass

import requests
import torch
import torch.nn.functional as F
from requests.exceptions import RequestException
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

def clean_secret(value: str) -> str:
    return value.strip().strip('"').strip("'")

def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise EnvironmentError(f"Missing required environment variable: {name}")
    return clean_secret(value)

def prompt_and_set_env(name: str):
    if not os.getenv(name):
        os.environ[name] = clean_secret(getpass(f"Enter {name}: "))
    else:
        os.environ[name] = clean_secret(os.environ[name])

for env_name in ["COURTLISTENER_API_TOKEN", "GOVINFO_API_KEY", "CONGRESS_API_KEY"]:
    prompt_and_set_env(env_name)

print("API keys loaded into environment variables.")


Enter COURTLISTENER_API_TOKEN: ··········
Enter GOVINFO_API_KEY: ··········
Enter CONGRESS_API_KEY: ··········
API keys loaded into environment variables.


In [6]:
def courtlistener_search(query, max_results=3):
    token = require_env("COURTLISTENER_API_TOKEN")
    url = "https://www.courtlistener.com/api/rest/v4/search/"
    headers = {
        "Authorization": f"Token {token}",
        "Accept": "application/json",
    }
    params = {
        "q": query,
        "type": "o",
    }

    summaries = []

    for attempt in range(3):
        try:
            res = requests.get(url, params=params, headers=headers, timeout=15)

            if res.status_code == 200:
                data = res.json()
                results = data.get("results", [])[:max_results]
                for item in results:
                    case_name = item.get("caseName") or item.get("caseNameFull") or "Unknown case"
                    court = item.get("court_citation_string") or item.get("court") or "Unknown court"
                    summaries.append(f"{case_name} ({court})")
                break

            if res.status_code == 429 or 500 <= res.status_code < 600:
                time.sleep(2 ** attempt)
                continue

            return {
                "tool": "CaseLawSearch",
                "input": query,
                "output": f"API error: HTTP {res.status_code} | {res.text[:300]}",
            }

        except RequestException as exc:
            if attempt == 2:
                return {
                    "tool": "CaseLawSearch",
                    "input": query,
                    "output": f"Network error: {exc}",
                }
            time.sleep(2 ** attempt)

    summaries = list(dict.fromkeys(summaries))
    if not summaries:
        summaries = ["No results found"]

    return {
        "tool": "CaseLawSearch",
        "input": query,
        "output": "; ".join(summaries),
    }


def statute_lookup(usc_title, section):
    api_key = require_env("GOVINFO_API_KEY")
    search_url = "https://api.govinfo.gov/search"
    params = {"api_key": api_key}
    payload = {
        "query": f'collection:uscode citation:"{usc_title} U.S.C. {section}"',
        "pageSize": "1",
        "offsetMark": "*",
        "resultLevel": "default",
    }

    try:
        res = requests.post(search_url, params=params, json=payload, timeout=10)
        res.raise_for_status()
        data = res.json()
        results = data.get("results", [])

        if not results:
            output = "Statute not found"
        else:
            item = results[0]
            title_text = item.get("title", "No title available")
            package_id = item.get("packageId", "")
            granule_id = item.get("granuleId", "")

            if granule_id:
                source = f"https://www.govinfo.gov/app/details/{package_id}/{granule_id}"
            elif package_id:
                source = f"https://www.govinfo.gov/app/details/{package_id}"
            else:
                source = "No link available"

            output = f"{title_text} | {source}"

    except requests.exceptions.RequestException as exc:
        output = f"Network error: {exc}"

    return {
        "tool": "StatuteLookup",
        "input": f"{usc_title} U.S.C. ? {section}",
        "output": output,
    }


def law_lookup(congress, law_number):
    api_key = require_env("CONGRESS_API_KEY")
    url = f"https://api.congress.gov/v3/law/{congress}/pub/{law_number}"
    params = {
        "api_key": api_key,
        "format": "json",
    }

    try:
        res = requests.get(url, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()

        bill = data.get("bill", {})
        title = bill.get("title", "No title")
        latest_action = bill.get("latestAction", {}).get("text", "No action info")
        output = f"{title} | Latest action: {latest_action}"

    except requests.exceptions.RequestException as exc:
        output = f"Network error: {exc}"

    return {
        "tool": "LawLookup",
        "input": f"Public Law {congress}-{law_number}",
        "output": output,
    }


def date_calculator(from_date, to_date):
    fmt = "%Y-%m-%d"
    try:
        d1 = datetime.strptime(from_date, fmt)
        d2 = datetime.strptime(to_date, fmt)
        result = f"{(d2 - d1).days} days"
    except ValueError:
        result = "Invalid date format"

    return {
        "tool": "DateCalculator",
        "input": f"{from_date} ? {to_date}",
        "output": result,
    }


In [7]:
def law_lookup(congress, law_number):
    api_key = require_env("CONGRESS_API_KEY")
    url = f"https://api.congress.gov/v3/law/{congress}/pub/{law_number}"
    params = {
        "api_key": api_key,
        "format": "json",
    }

    try:
        res = requests.get(url, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()

        bill = data.get("bill", {})
        title = bill.get("title", "No title")
        latest_action = bill.get("latestAction", {}).get("text", "No action info")
        output = f"{title} | Latest action: {latest_action}"

    except requests.exceptions.RequestException as e:
        output = f"Network error: {e}"

    return {
        "tool": "LawLookup",
        "input": f"Public Law {congress}-{law_number}",
        "output": output,
    }


In [8]:
DEHYPHEN = lambda t: re.sub(r'(\w+)-\s+(\w)', lambda m: m.group(1) + m.group(2), t)
MIN_LEN  = 60

def looks_legal(text):
    t = text.lower()
    if "____" in text or len(text.split()) < 8:
        return False
    signals = [
        " v. ", "plaintiff", "defendant", "court", "statute",
        "pursuant", "u.s.c", "§", "section", "held ", "ruling",
        "judgment", "appeal", "liability", "petitioner",
        "respondent", "affirmed", "reversed", "remanded", "cfr",
    ]
    return any(s in t for s in signals)

texts = []
for raw in tqdm(texts_raw[:5000], desc="Extracting sentences"):
    if not isinstance(raw, str):
        continue
    raw = DEHYPHEN(raw)
    for sentence in raw.replace("\n", " ").split(". "):
        sentence = sentence.strip()
        if len(sentence) >= MIN_LEN and looks_legal(sentence):
            texts.append(sentence)
    if len(texts) >= 50000:
        break

print(f"Total sentences: {len(texts)}")

Extracting sentences:  35%|███▍      | 1731/5000 [00:04<00:07, 417.42it/s]

Total sentences: 50007


In [9]:
SENTENCE_INSERT_RE = re.compile(r"\.\s+(?=[A-Z])")
FALLBACK_INSERT_PATTERNS = [
    re.compile(r";\s+"),
    re.compile(r":\s+"),
]
TOOL_SEPARATOR = "->"
MIN_CONTINUATION_CHARS = 10
TOOLFORMER_THRESHOLD = 0.0
FALLBACK_RATIO = 1.05


In [10]:
def split_at_tool(augmented_text: str):
    pattern = rf"(\[.*?\s{re.escape(TOOL_SEPARATOR)}\s.*?\])"
    match = re.search(pattern, augmented_text)
    if not match:
        return None, None, None
    prefix = augmented_text[:match.start()]
    tool_block = augmented_text[match.start():match.end()]
    continuation = augmented_text[match.end():]
    return prefix, tool_block, continuation


In [11]:
def insert_tool_midpoint(text, tool_call, output):
    tool_block = f" [{tool_call} {TOOL_SEPARATOR} {output}]"

    clause_patterns = [
        r",\s+which\s+",
        r",\s+that\s+",
        r",\s+where\s+",
        r",\s+when\s+",
        r",\s+because\s+",
        r";\s+",
        r",\s+",
    ]

    for pattern in clause_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            insert_at = match.start()
            prefix = text[:insert_at]
            continuation = text[insert_at:]
            if len(prefix.split()) >= 8 and len(continuation.split()) >= 8:
                return prefix + tool_block + continuation

    sentence_match = SENTENCE_INSERT_RE.search(text)
    if sentence_match and sentence_match.start() > 20:
        insert_at = sentence_match.start() + 1
        return text[:insert_at] + tool_block + text[insert_at:]

    for pattern in FALLBACK_INSERT_PATTERNS:
        fallback = pattern.search(text)
        if fallback and fallback.start() > 20:
            return text[:fallback.start()] + tool_block + text[fallback.start():]

    words = text.split()
    if len(words) >= 20:
        mid = len(words) // 2
        prefix = " ".join(words[:mid])
        continuation = " " + " ".join(words[mid:])
        return prefix + tool_block + continuation

    return text + tool_block


In [12]:
import time
SECTION_TO_TITLE = {
    "441": "52", "110": "11", "1": "52", "2": "52",
    "30": "52",  "301": "52", "431": "52", "432": "52",
    "433": "52", "434": "52", "437": "52", "438": "52",
}

LANDMARK_NAMES = [
    "Buckley", "McCutcheon", "Citizens United", "McConnell",
    "Randall", "Shrink Missouri", "Colorado Republican",
    "Nixon", "Bellotti", "Austin", "Valeo", "Davis",
    "Wisconsin Right", "Beaumont", "Burson", "Eu",
]

def detect_tool(text):
    t = text.lower()
    if re.search(r"§\s*\d+|u\.s\.c\.?\s*[§ss]?\s*\d+|\bcfr\s*[§ss]?\s*\d+|\bsection\s+\d+", t):
        return "StatuteLookup"
    if "public law" in t:
        return "LawLookup"
    if re.search(r'[A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)?\s+v\.\s+[A-Z][a-zA-Z]', text):
        return "CaseLawSearch"
    if any(name in text for name in LANDMARK_NAMES):
        return "CaseLawSearch"
    if re.search(r"\d{4}-\d{2}-\d{2}|\(\d{4}\)|\bin\s+\d{4}\b", t):
        return "DateCalculator"
    return None

def extract_statute(text):
    # Full U.S.C.
    match = re.search(r"(\d+)\s*U\.S\.C\.?\s*[§Ss]?\s*(\d+)", text)
    if match:
        return match.groups()
    # CFR
    match = re.search(r"(\d+)\s*CFR\s*[§Ss]?\s*(\d+)", text)
    if match:
        return match.groups()
    # Bare § with title lookup
    match = re.search(r"§\s*(\d+)", text)
    if match:
        section = match.group(1)
        for prefix_len in [3, 2, 1]:
            prefix = section[:prefix_len]
            if prefix in SECTION_TO_TITLE:
                return (SECTION_TO_TITLE[prefix], section)
        return ("52", section)
    # Section N
    match = re.search(r"[Ss]ection\s+(\d+)", text)
    if match:
        return ("52", match.group(1))
    return None

def extract_case_name(text):
    # Full "X v. Y"
    match = re.search(
        r'((?:[A-Z][a-zA-Z&.,]+(?:\s[A-Z][a-zA-Z&.,]+)*)\s+v\.\s+(?:[A-Z][a-zA-Z&.,]+(?:\s[A-Z][a-zA-Z&.,]+)*))',
        text
    )
    if match:
        return match.group(1)
    # Landmark single name
    for name in LANDMARK_NAMES:
        if name in text:
            return name
    return None

def extract_dates(text):
    matches = re.findall(r"\d{4}-\d{2}-\d{2}", text)
    if len(matches) >= 2:
        return sorted(matches)
    years = list(dict.fromkeys(re.findall(r"\b((?:19|20)\d{2})\b", text)))
    if len(years) >= 2:
        y = sorted(years)
        return [f"{y[0]}-01-01", f"{y[1]}-01-01"]
    return None

def extract_law(text):
    match = re.search(r"Public Law (\d+)-(\d+)", text)
    return match.groups() if match else None

def build_tool_call(tool, text):
    if tool == "CaseLawSearch":
        case = extract_case_name(text)
        if case:
            return f'CaseLawSearch("{case}")', case
    if tool == "StatuteLookup":
        statute = extract_statute(text)
        if statute:
            return f'StatuteLookup("{statute[0]}", "{statute[1]}")', statute
    if tool == "LawLookup":
        law = extract_law(text)
        if law:
            return f'LawLookup("{law[0]}", "{law[1]}")', law
    if tool == "DateCalculator":
        dates = extract_dates(text)
        if dates:
            return f'DateCalculator("{dates[0]}", "{dates[1]}")', dates
    return None, None


def run_tool(tool, args):
    try:
        if tool == "CaseLawSearch":
            result = courtlistener_search(args)
            time.sleep(0.5)   # CourtListener rate limit safety
            return result
        if tool == "StatuteLookup":
            return statute_lookup(args[0], args[1])
        if tool == "LawLookup":
            return law_lookup(args[0], args[1])
        if tool == "DateCalculator":
            return date_calculator(args[0], args[1])
    except Exception as e:
        return {"output": f"API error: {e}"}
    return {"output": "No result"}

In [13]:
def generate_candidates(texts, debug=False):
    augmented_data = []
    skipped_no_tool = 0
    skipped_extract = 0
    skipped_bad_output = 0

    for text in tqdm(texts, desc="Generating candidates"):
        tool = detect_tool(text)
        if not tool:
            skipped_no_tool += 1
            continue

        tool_call, args = build_tool_call(tool, text)
        if not tool_call:
            skipped_extract += 1
            if debug:
                print(f"[SKIP extract] {tool}: {text[:80]!r}")
            continue

        tool_result = run_tool(tool, args)
        output = tool_result.get("output", "").strip()

        if any(x in output.lower() for x in ["error", "api error", "network error", "http error"]):
            skipped_bad_output += 1
            if debug:
                print(f"[SKIP output] {tool_call}: {output!r}")
            continue

        output = output.split(";")[0].strip()

        augmented_text = insert_tool_midpoint(text, tool_call, output)

        augmented_data.append({
            "original_text": text,
            "augmented_text": augmented_text,
            "tool": tool,
            "tool_call": tool_call,
            "tool_output": output,
        })

    print(f"\n--- generate_candidates summary ---")
    print(f"  Input:            {len(texts)}")
    print(f"  Skipped(no tool): {skipped_no_tool}")
    print(f"  Skipped(extract): {skipped_extract}")
    print(f"  Skipped(output):  {skipped_bad_output}")
    print(f"  Candidates:       {len(augmented_data)}")

    tool_counts = Counter(d["tool"] for d in augmented_data)
    print(f"\n--- Tool breakdown ---")
    for tool_name, count in tool_counts.items():
        print(f"  {tool_name}: {count}")

    return augmented_data


In [14]:
MODEL_NAME = "distilgpt2"
BATCH_SIZE = 32
MAX_LENGTH = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Scoring device: {DEVICE}")
print(f"Scoring model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
if DEVICE == "cuda":
    model = model.half()
model.eval()


def compute_losses_batch(texts_batch):
    inputs = tokenizer(
        texts_batch,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_attention_mask=True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        logits = outputs.logits

    losses = []
    for i in range(len(texts_batch)):
        labels = inputs["input_ids"][i].clone()
        labels[inputs["attention_mask"][i] == 0] = -100
        shift_logits = logits[i, :-1, :].float()
        shift_labels = labels[1:]
        loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)
        losses.append(loss.item())
    return losses


def get_continuation_loss(prefix: str, continuation: str) -> float:
    full_text = prefix + continuation
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    cont_len = len(full_ids) - len(prefix_ids)

    if cont_len <= 0:
        return float("inf")

    if len(full_ids) > MAX_LENGTH:
        full_ids = full_ids[-MAX_LENGTH:]
        continuation_start = max(0, len(full_ids) - cont_len)
    else:
        continuation_start = len(prefix_ids)

    input_ids = torch.tensor([full_ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits[0].float()

    labels = torch.full((len(full_ids),), -100, dtype=torch.long).to(DEVICE)
    labels[continuation_start:] = input_ids[0][continuation_start:]

    shift_logits = logits[:-1]
    shift_labels = labels[1:]

    loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)
    return loss.item()


def filter_by_loss_batch(samples, debug=False):
    filtered = []
    debug_count = 0
    high_loss_threshold = 4.5

    for batch_start in tqdm(range(0, len(samples), BATCH_SIZE), desc="Batch pre-filter"):
        batch = samples[batch_start: batch_start + BATCH_SIZE]
        originals = [s["original_text"] for s in batch]
        augmented = [s["augmented_text"] for s in batch]

        try:
            losses_orig = compute_losses_batch(originals)
            losses_aug = compute_losses_batch(augmented)
        except Exception as exc:
            print(f"[WARN] Batch failed: {exc}")
            continue

        for i, sample in enumerate(batch):
            loss_original = losses_orig[i]
            loss_augmented = losses_aug[i]
            keep = (
                loss_augmented <= loss_original * 1.20
                or loss_original >= high_loss_threshold
            )

            if debug and debug_count < 5:
                print(f"\nOriginal  ({loss_original:.4f}): {sample['original_text'][:80]}")
                print(f"Augmented ({loss_augmented:.4f}): {sample['augmented_text'][:80]}")
                print(f"keep={keep}")
                debug_count += 1

            if keep:
                filtered.append(sample)

    print(f"\nBatch pre-filtered: {len(filtered)} / {len(samples)}")
    return filtered


Scoring device: cuda
Scoring model: distilgpt2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [16]:
TARGET_INPUT_SIZE = 3000
RUN_DEBUG = False

print(f"Generating candidates from {TARGET_INPUT_SIZE} legal sentences...")
candidates = generate_candidates(texts[:TARGET_INPUT_SIZE], debug=RUN_DEBUG)


Generating candidates from 3000 legal sentences...


Generating candidates: 100%|██████████| 3000/3000 [03:23<00:00, 14.74it/s]


--- generate_candidates summary ---
  Input:            3000
  Skipped(no tool): 2481
  Skipped(extract): 67
  Skipped(output):  0
  Candidates:       452

--- Tool breakdown ---
  StatuteLookup: 401
  CaseLawSearch: 46
  DateCalculator: 5


In [17]:
def filter_by_loss(samples, debug=False):
    filtered = []
    debug_count = 0

    for sample in tqdm(samples, desc="Toolformer filtering"):
        augmented = sample["augmented_text"]
        original = sample["original_text"]

        prefix, tool_block, continuation = split_at_tool(augmented)
        if prefix is None:
            continue

        try:
            if len(continuation.strip()) > MIN_CONTINUATION_CHARS:
                loss_without = get_continuation_loss(prefix, continuation)
                loss_with = get_continuation_loss(prefix + tool_block, continuation)
                improvement = loss_without - loss_with
                keep = improvement > TOOLFORMER_THRESHOLD

                if debug and debug_count < 8:
                    print(f"\n--- SAMPLE ---")
                    print(f"  Prefix:       {prefix[:70]!r}")
                    print(f"  Tool:         {tool_block[:70]!r}")
                    print(f"  Continuation: {continuation[:70]!r}")
                    print(f"  Loss without: {loss_without:.4f}")
                    print(f"  Loss with:    {loss_with:.4f}")
                    print(f"  Improvement:  {improvement:+.4f}")
                    print(f"  Keep:         {keep}")
                    debug_count += 1
            else:
                loss_orig = get_continuation_loss("", original)
                loss_aug = get_continuation_loss("", augmented)
                keep = loss_aug <= loss_orig * FALLBACK_RATIO

                if debug and debug_count < 8:
                    print(f"\n--- SAMPLE (fallback) ---")
                    print(f"  Loss orig: {loss_orig:.4f} | Loss aug: {loss_aug:.4f}")
                    print(f"  Keep: {keep}")
                    debug_count += 1

        except Exception as exc:
            print(f"[WARN] {exc}")
            continue

        if keep:
            filtered.append(sample)

    print(f"\nFiltered: {len(filtered)} / {len(samples)}")
    print(f"Pass rate: {len(filtered) / len(samples) * 100:.1f}%")
    return filtered


In [18]:
pre_filtered = filter_by_loss_batch(candidates, debug=False)
filtered = filter_by_loss(pre_filtered, debug=True)

print(f"Candidates: {len(candidates)}")
print(f"Pre-filtered: {len(pre_filtered)}")
print(f"Final filtered: {len(filtered)}")


Batch pre-filter: 100%|██████████| 15/15 [00:03<00:00,  4.55it/s]



Batch pre-filtered: 388 / 452


Toolformer filtering:   2%|▏         | 7/388 [00:00<00:06, 63.49it/s]


--- SAMPLE ---
  Prefix:       '§441a(a)(2).3 The base limits apply with equal force to contributions '
  Tool:         '[StatuteLookup("52", "441") -> Statute not found]'
  Continuation: ' any way earmarked or otherwise directed through an intermediary or co'
  Loss without: 4.2972
  Loss with:    4.9625
  Improvement:  -0.6653
  Keep:         False

--- SAMPLE (fallback) ---
  Loss orig: 5.3988 | Loss aug: 5.3007
  Keep: True

--- SAMPLE ---
  Prefix:       'By contrast, the Court concluded that contribution limits impose a les'
  Tool:         '[ ] the symbolic expression of support [CaseLawSearch("Buckley") -> Ma'
  Continuation: ' evidenced by a contribution but do[ ] not in any way infringe the con'
  Loss without: 4.0349
  Loss with:    4.0200
  Improvement:  +0.0149
  Keep:         True

--- SAMPLE ---
  Prefix:       'Buckley held that the Government’s interest in preventing quid pro quo'
  Tool:         '[CaseLawSearch("Buckley") -> Marilyn Buckley v. Freddie Buckley (Ark. '

Toolformer filtering: 100%|██████████| 388/388 [00:06<00:00, 64.21it/s]


Filtered: 206 / 388
Pass rate: 53.1%
Candidates: 452
Pre-filtered: 388
Final filtered: 206


In [19]:
import json
from pathlib import Path

OUTPUT_DIR = Path("toolformer_legal_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

training_examples = []
for sample in filtered:
    training_examples.append({
        "text": sample["augmented_text"],
        "original_text": sample["original_text"],
        "tool": sample["tool"],
        "tool_call": sample["tool_call"],
        "tool_output": sample["tool_output"],
    })

with open(OUTPUT_DIR / "toolformer_legal_train_full.json", "w", encoding="utf-8") as f:
    json.dump(training_examples, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "toolformer_legal_train_full.jsonl", "w", encoding="utf-8") as f:
    for row in training_examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(OUTPUT_DIR / "toolformer_legal_train_full.txt", "w", encoding="utf-8") as f:
    for row in training_examples:
        f.write(row["text"].strip() + "\n")

print(f"Saved {len(training_examples)} filtered examples to {OUTPUT_DIR}")


Saved 206 filtered examples to toolformer_legal_outputs


In [20]:
import random
from collections import Counter

SPLIT_SEED = 42
random.seed(SPLIT_SEED)
shuffled_examples = training_examples[:]
random.shuffle(shuffled_examples)

total = len(shuffled_examples)
train_end = max(1, int(total * 0.8)) if total >= 3 else total
val_end = max(train_end + 1, int(total * 0.9)) if total >= 10 else train_end
val_end = min(val_end, total)

train_split = shuffled_examples[:train_end]
val_split = shuffled_examples[train_end:val_end]
test_split = shuffled_examples[val_end:]

splits = {
    "train": train_split,
    "dev": val_split,
    "test": test_split,
}

for split_name, rows in splits.items():
    with open(OUTPUT_DIR / f"{split_name}.json", "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    with open(OUTPUT_DIR / f"{split_name}.jsonl", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    with open(OUTPUT_DIR / f"{split_name}.txt", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(row["text"].strip() + "\n")

print("Split sizes:")
print({k: len(v) for k, v in splits.items()})
print("Tool counts in filtered set:", Counter(x["tool"] for x in training_examples))
print(f"Artifacts written to: {OUTPUT_DIR.resolve()}")


Split sizes:
{'train': 164, 'dev': 21, 'test': 21}
Tool counts in filtered set: Counter({'StatuteLookup': 173, 'CaseLawSearch': 29, 'DateCalculator': 4})
Artifacts written to: /content/toolformer_legal_outputs


In [21]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.9 MB/s eta 0:00:00


In [25]:
!pip install -q pyarrow==15.0.2 datasets==2.19.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.3.0 requires datasets>=4.7.0, but you have datasets 2.19.1 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version

In [26]:
import json
from datasets import Dataset

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_rows = load_json("toolformer_legal_outputs/train.json")
dev_rows = load_json("toolformer_legal_outputs/dev.json")
test_rows = load_json("toolformer_legal_outputs/test.json")

print(len(train_rows), len(dev_rows), len(test_rows))
print(train_rows[0].keys())


<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:488: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


176 21 21
dict_keys(['text', 'original_text', 'tool', 'tool_call', 'tool_output'])


In [27]:
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_rows = load_json("toolformer_legal_outputs/train.json")
dev_rows = load_json("toolformer_legal_outputs/dev.json")
test_rows = load_json("toolformer_legal_outputs/test.json")

print(len(train_rows), len(dev_rows), len(test_rows))
print(train_rows[0].keys())


176 21 21
dict_keys(['text', 'original_text', 'tool', 'tool_call', 'tool_output'])


In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.config.pad_token_id = tokenizer.pad_token_id


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=256):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        text = self.rows[idx]["text"]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = item["input_ids"].clone()
        return item

train_dataset = TextDataset(train_rows, tokenizer)
dev_dataset = TextDataset(dev_rows, tokenizer)
test_dataset = TextDataset(test_rows, tokenizer)


In [30]:
!pip install -q peft accelerate transformers


In [32]:
!pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [33]:
!pip install -q "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00


In [34]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="toolformer_legal_lora_run1",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    load_best_model_at_end=True,
)


In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
)

trainer.train()


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,6.322324,4.718637
2,1.353903,1.460191
3,1.397936,1.400245
4,1.361126,1.366445
5,1.352104,1.355540


TrainOutput(global_step=220, training_loss=2.755925824425437, metrics={'train_runtime': 30.8457, 'train_samples_per_second': 28.529, 'train_steps_per_second': 7.132, 'total_flos': 57684598456320.0, 'train_loss': 2.755925824425437, 'epoch': 5.0})

In [38]:
eval_results = trainer.evaluate(test_dataset)
print(eval_results)

{'eval_loss': 1.247624158859253, 'eval_runtime': 0.3782, 'eval_samples_per_second': 55.521, 'eval_steps_per_second': 15.863, 'epoch': 5.0}


In [39]:
model.save_pretrained("toolformer_legal_lora/final_adapter")
tokenizer.save_pretrained("toolformer_legal_lora/final_adapter")
print("Saved fine-tuned adapter.")

Saved fine-tuned adapter.


In [42]:
import math
test_results = trainer.evaluate(test_dataset)
test_loss = test_results["eval_loss"]
test_ppl = math.exp(test_loss)

print("Test loss:", test_loss)
print("Test perplexity:", test_ppl)


Test loss: 1.247624158859253
Test perplexity: 3.4820603001032784


In [43]:
heldout_eval = [
    {
        "prompt": "What case established the framework for contribution and expenditure limits in campaign finance law?",
        "expected_tool": "CaseLawSearch",
        "expected_args": ["Buckley v. Valeo"],
        "reference_answer": "Buckley v. Valeo"
    },
    {
        "prompt": "Which Supreme Court case held that independent expenditures by corporations are protected political speech?",
        "expected_tool": "CaseLawSearch",
        "expected_args": ["Citizens United v. Federal Election Commission"],
        "reference_answer": "Citizens United v. Federal Election Commission"
    },
    {
        "prompt": "What case is commonly cited for the constitutional right to abortion recognized in 1973?",
        "expected_tool": "CaseLawSearch",
        "expected_args": ["Roe v. Wade"],
        "reference_answer": "Roe v. Wade"
    },
    {
        "prompt": "Which case defended broader anticorruption rationales in campaign finance law before McCutcheon narrowed them?",
        "expected_tool": "CaseLawSearch",
        "expected_args": ["McConnell v. Federal Election Commission"],
        "reference_answer": "McConnell v. Federal Election Commission"
    },

    {
        "prompt": "Look up the statutory definitions in 52 U.S.C. 30101.",
        "expected_tool": "StatuteLookup",
        "expected_args": ["52", "30101"],
        "reference_answer": "52 U.S.C. 30101"
    },
    {
        "prompt": "Find the regulation referenced in 11 CFR 110.1.",
        "expected_tool": "StatuteLookup",
        "expected_args": ["11", "110.1"],
        "reference_answer": "11 CFR 110.1"
    },
    {
        "prompt": "What does section 30116 of Title 52 cover?",
        "expected_tool": "StatuteLookup",
        "expected_args": ["52", "30116"],
        "reference_answer": "52 U.S.C. 30116"
    },
    {
        "prompt": "Retrieve the campaign finance disclosure provisions in 52 U.S.C. 30104.",
        "expected_tool": "StatuteLookup",
        "expected_args": ["52", "30104"],
        "reference_answer": "52 U.S.C. 30104"
    },

    {
        "prompt": "What is Public Law 104-104?",
        "expected_tool": "LawLookup",
        "expected_args": ["104", "104"],
        "reference_answer": "Telecommunications Act of 1996"
    },
    {
        "prompt": "Identify Public Law 111-148.",
        "expected_tool": "LawLookup",
        "expected_args": ["111", "148"],
        "reference_answer": "Patient Protection and Affordable Care Act"
    },
    {
        "prompt": "What law corresponds to Public Law 107-56?",
        "expected_tool": "LawLookup",
        "expected_args": ["107", "56"],
        "reference_answer": "USA PATRIOT Act"
    },
    {
        "prompt": "Find the title of Public Law 109-246.",
        "expected_tool": "LawLookup",
        "expected_args": ["109", "246"],
        "reference_answer": "Voting Rights Act Reauthorization and Amendments Act of 2006"
    },

    {
        "prompt": "How many days passed between 1973-01-22 and 1992-06-29?",
        "expected_tool": "DateCalculator",
        "expected_args": ["1973-01-22", "1992-06-29"],
        "reference_answer": "7107 days"
    },
    {
        "prompt": "Compute the number of days from 2010-01-21 to 2014-04-02.",
        "expected_tool": "DateCalculator",
        "expected_args": ["2010-01-21", "2014-04-02"],
        "reference_answer": "1532 days"
    },
    {
        "prompt": "What is the day difference between 1976-01-30 and 2014-04-02?",
        "expected_tool": "DateCalculator",
        "expected_args": ["1976-01-30", "2014-04-02"],
        "reference_answer": "13942 days"
    },
    {
        "prompt": "Calculate the interval from 2003-12-10 to 2010-01-21.",
        "expected_tool": "DateCalculator",
        "expected_args": ["2003-12-10", "2010-01-21"],
        "reference_answer": "2234 days"
    },

    {
        "prompt": "Summarize the importance of judicial independence in constitutional adjudication.",
        "expected_tool": "NONE",
        "expected_args": [],
        "reference_answer": "No tool needed"
    },
    {
        "prompt": "Explain why stare decisis matters in Supreme Court decisionmaking.",
        "expected_tool": "NONE",
        "expected_args": [],
        "reference_answer": "No tool needed"
    },
    {
        "prompt": "Discuss the relationship between representation and democratic accountability.",
        "expected_tool": "NONE",
        "expected_args": [],
        "reference_answer": "No tool needed"
    },
    {
        "prompt": "Describe the role of dissenting opinions in constitutional law.",
        "expected_tool": "NONE",
        "expected_args": [],
        "reference_answer": "No tool needed"
    },
]

print("Held-out examples:", len(heldout_eval))


Held-out examples: 20


In [44]:
import re
import json
from collections import Counter

def normalize_text(text):
    return re.sub(r"\s+", " ", text.strip().lower())

def exact_match(a, b):
    return normalize_text(a) == normalize_text(b)

def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def run_tool_from_call(tool_name, args):
    if tool_name == "CaseLawSearch":
        return courtlistener_search(args[0])
    if tool_name == "StatuteLookup":
        return statute_lookup(args[0], args[1])
    if tool_name == "LawLookup":
        return law_lookup(args[0], args[1])
    if tool_name == "DateCalculator":
        return date_calculator(args[0], args[1])
    return {"output": "No tool used"}


In [45]:
from transformers import pipeline

base_generator = pipeline(
    "text-generation",
    model=BASE_MODEL,
    tokenizer=tokenizer,
    max_new_tokens=64,
)

def generate_answer(generator, prompt):
    out = generator(prompt, max_new_tokens=64, do_sample=False)[0]["generated_text"]
    return out[len(prompt):].strip()

base_results = []
for example in heldout_eval:
    pred = generate_answer(base_generator, example["prompt"] + "\nAnswer:")
    base_results.append({
        "prompt": example["prompt"],
        "prediction": pred,
        "reference": example["reference_answer"],
        "em": exact_match(pred, example["reference_answer"]),
        "f1": token_f1(pred, example["reference_answer"]),
    })

base_em = sum(r["em"] for r in base_results) / len(base_results)
base_f1 = sum(r["f1"] for r in base_results) / len(base_results)

print("Base EM:", round(base_em, 3))
print("Base F1:", round(base_f1, 3))


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Bot

Base EM: 0.0
Base F1: 0.019


In [46]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

ft_base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
ft_tokenizer = AutoTokenizer.from_pretrained("toolformer_legal_lora/final_adapter")
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

ft_model = PeftModel.from_pretrained(ft_base_model, "toolformer_legal_lora/final_adapter")

ft_generator = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    max_new_tokens=64,
)

ft_results = []
for example in heldout_eval:
    pred = generate_answer(ft_generator, example["prompt"] + "\nAnswer:")
    ft_results.append({
        "prompt": example["prompt"],
        "prediction": pred,
        "reference": example["reference_answer"],
        "em": exact_match(pred, example["reference_answer"]),
        "f1": token_f1(pred, example["reference_answer"]),
    })

ft_em = sum(r["em"] for r in ft_results) / len(ft_results)
ft_f1 = sum(r["f1"] for r in ft_results) / len(ft_results)

print("Fine-tuned EM:", round(ft_em, 3))
print("Fine-tuned F1:", round(ft_f1, 3))


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for mo

Fine-tuned EM: 0.0
Fine-tuned F1: 0.0


In [47]:
def tool_augmented_answer(prompt):
    pred_tool, pred_args, pred_call, raw_completion = propose_tool_call(prompt)

    if pred_tool is None or pred_tool == "NONE":
        return {
            "tool": "NONE",
            "args": [],
            "answer": "No tool needed",
            "tool_output": "",
        }

    tool_result = run_tool_from_call(pred_tool, pred_args)
    tool_output = tool_result.get("output", "")

    final_answer = f"{pred_call} -> {tool_output}"
    return {
        "tool": pred_tool,
        "args": pred_args,
        "answer": final_answer,
        "tool_output": tool_output,
    }

tool_enabled_results = []
for example in heldout_eval:
    result = tool_augmented_answer(example["prompt"])
    pred = result["tool_output"] if result["tool_output"] else result["answer"]

    tool_enabled_results.append({
        "prompt": example["prompt"],
        "prediction": pred,
        "reference": example["reference_answer"],
        "tool": result["tool"],
        "args": result["args"],
        "em": exact_match(pred, example["reference_answer"]),
        "f1": token_f1(pred, example["reference_answer"]),
    })

tool_em = sum(r["em"] for r in tool_enabled_results) / len(tool_enabled_results)
tool_f1 = sum(r["f1"] for r in tool_enabled_results) / len(tool_enabled_results)

print("Fine-tuned + tools EM:", round(tool_em, 3))
print("Fine-tuned + tools F1:", round(tool_f1, 3))


NameError: name 'propose_tool_call' is not defined

In [48]:
import ast
import json
import re

LANDMARK_NAMES = [
    "Buckley", "McCutcheon", "Citizens United", "McConnell",
    "Randall", "Shrink Missouri", "Colorado Republican",
    "Nixon", "Bellotti", "Austin", "Valeo", "Davis",
    "Wisconsin Right", "Beaumont", "Burson", "Eu",
]

SECTION_TO_TITLE = {
    "441": "52", "110": "11", "1": "52", "2": "52",
    "30": "52", "301": "52", "431": "52", "432": "52",
    "433": "52", "434": "52", "437": "52", "438": "52",
}

def extract_case_name(text):
    match = re.search(
        r"([A-Z][A-Za-z&.,'-]+(?:\s+[A-Z][A-Za-z&.,'-]+)*\s+v\.\s+[A-Z][A-Za-z&.,'-]+(?:\s+[A-Z][A-Za-z&.,'-]+)*)",
        text
    )
    if match:
        return match.group(1)
    for name in LANDMARK_NAMES:
        if name in text:
            return name
    return None

def extract_statute(text):
    match = re.search(r"(\d+)\s*U\.S\.C\.?\s*[§Ss]?\s*(\d+)", text)
    if match:
        return [match.group(1), match.group(2)]

    match = re.search(r"(\d+)\s*CFR\s*[§Ss]?\s*(\d+(?:\.\d+)*)", text)
    if match:
        return [match.group(1), match.group(2)]

    match = re.search(r"§\s*(\d+(?:\.\d+)*)", text)
    if match:
        section = match.group(1)
        for prefix_len in [3, 2, 1]:
            prefix = section[:prefix_len]
            if prefix in SECTION_TO_TITLE:
                return [SECTION_TO_TITLE[prefix], section]
        return ["52", section]

    match = re.search(r"[Ss]ection\s+(\d+)", text)
    if match:
        return ["52", match.group(1)]

    return None

def extract_law(text):
    match = re.search(r"Public Law\s+(\d+)-(\d+)", text)
    if match:
        return [match.group(1), match.group(2)]
    return None

def extract_dates(text):
    matches = re.findall(r"\d{4}-\d{2}-\d{2}", text)
    if len(matches) >= 2:
        return sorted(matches[:2])

    years = re.findall(r"\b(?:19|20)\d{2}\b", text)
    years = list(dict.fromkeys(years))
    if len(years) >= 2:
        years = sorted(years[:2])
        return [f"{years[0]}-01-01", f"{years[1]}-01-01"]

    return None

def canonical_tool_call(tool_name, args):
    return f"{tool_name}(" + ", ".join(json.dumps(str(arg)) for arg in args) + ")"

def route_tool_heuristic(text):
    lower = text.lower()

    if extract_law(text):
        return "LawLookup"
    if "u.s.c" in lower or "cfr" in lower or "§" in text or extract_statute(text):
        return "StatuteLookup"
    if extract_dates(text):
        return "DateCalculator"
    if extract_case_name(text) or any(name.lower() in lower for name in LANDMARK_NAMES):
        return "CaseLawSearch"
    return "NONE"

def propose_tool_call(text):
    tool_name = route_tool_heuristic(text)

    if tool_name == "NONE":
        return None, [], None, "NONE"

    if tool_name == "CaseLawSearch":
        case = extract_case_name(text)
        if case:
            args = [case]
            return tool_name, args, canonical_tool_call(tool_name, args), "heuristic_case"
        return None, [], None, "heuristic_case_failed"

    if tool_name == "StatuteLookup":
        statute = extract_statute(text)
        if statute:
            return tool_name, statute, canonical_tool_call(tool_name, statute), "heuristic_statute"
        return None, [], None, "heuristic_statute_failed"

    if tool_name == "LawLookup":
        law = extract_law(text)
        if law:
            return tool_name, law, canonical_tool_call(tool_name, law), "heuristic_law"
        return None, [], None, "heuristic_law_failed"

    if tool_name == "DateCalculator":
        dates = extract_dates(text)
        if dates:
            return tool_name, dates, canonical_tool_call(tool_name, dates), "heuristic_date"
        return None, [], None, "heuristic_date_failed"

    return None, [], None, "NONE"


In [49]:
print(propose_tool_call("What is Public Law 104-104?"))
print(propose_tool_call("Look up 52 U.S.C. 30101."))
print(propose_tool_call("How many days passed between 1973-01-22 and 1992-06-29?"))
print(propose_tool_call("Which case established campaign finance contribution limits?"))


('LawLookup', ['104', '104'], 'LawLookup("104", "104")', 'heuristic_law')
('StatuteLookup', ['52', '30101'], 'StatuteLookup("52", "30101")', 'heuristic_statute')
('DateCalculator', ['1973-01-22', '1992-06-29'], 'DateCalculator("1973-01-22", "1992-06-29")', 'heuristic_date')
(None, [], None, 'NONE')


In [50]:
def evaluate_toolformer_routing(eval_set):
    rows = []

    for example in eval_set:
        prompt = example["prompt"]
        expected_tool = example["expected_tool"]
        expected_args = example["expected_args"]

        pred_tool, pred_args, pred_call, raw_completion = propose_tool_call(prompt)

        if pred_tool is None:
            pred_tool = "NONE"
            pred_args = []

        tool_correct = int(pred_tool == expected_tool)
        args_correct = int(pred_args == expected_args) if expected_tool != "NONE" else int(pred_tool == "NONE")

        rows.append({
            "prompt": prompt,
            "expected_tool": expected_tool,
            "pred_tool": pred_tool,
            "tool_correct": tool_correct,
            "expected_args": expected_args,
            "pred_args": pred_args,
            "args_correct": args_correct,
            "tool_call": pred_call,
            "raw_completion": raw_completion,
            "reference_answer": example["reference_answer"],
        })

    tool_acc = sum(r["tool_correct"] for r in rows) / len(rows)
    arg_acc = sum(r["args_correct"] for r in rows) / len(rows)

    print("Tool selection accuracy:", round(tool_acc, 3))
    print("Argument accuracy:", round(arg_acc, 3))
    return rows

tool_eval_rows = evaluate_toolformer_routing(heldout_eval)


Tool selection accuracy: 0.85
Argument accuracy: 0.8


In [51]:
def run_tool_from_call(tool_name, args):
    if tool_name == "CaseLawSearch":
        return courtlistener_search(args[0])
    if tool_name == "StatuteLookup":
        return statute_lookup(args[0], args[1])
    if tool_name == "LawLookup":
        return law_lookup(args[0], args[1])
    if tool_name == "DateCalculator":
        return date_calculator(args[0], args[1])
    return {"output": "No tool used"}


In [52]:
import re
from collections import Counter

def normalize_text(text):
    return re.sub(r"\s+", " ", text.strip().lower())

def exact_match(a, b):
    return normalize_text(a) == normalize_text(b)

def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


In [53]:
def tool_augmented_answer(prompt):
    pred_tool, pred_args, pred_call, raw_completion = propose_tool_call(prompt)

    if pred_tool is None or pred_tool == "NONE":
        return {
            "tool": "NONE",
            "args": [],
            "answer": "No tool needed",
            "tool_output": "",
        }

    tool_result = run_tool_from_call(pred_tool, pred_args)
    tool_output = tool_result.get("output", "")

    return {
        "tool": pred_tool,
        "args": pred_args,
        "answer": f"{pred_call} -> {tool_output}",
        "tool_output": tool_output,
    }


In [54]:
tool_enabled_results = []
for example in heldout_eval:
    result = tool_augmented_answer(example["prompt"])
    pred = result["tool_output"] if result["tool_output"] else result["answer"]

    tool_enabled_results.append({
        "prompt": example["prompt"],
        "prediction": pred,
        "reference": example["reference_answer"],
        "tool": result["tool"],
        "args": result["args"],
        "em": exact_match(pred, example["reference_answer"]),
        "f1": token_f1(pred, example["reference_answer"]),
    })

tool_em = sum(r["em"] for r in tool_enabled_results) / len(tool_enabled_results)
tool_f1 = sum(r["f1"] for r in tool_enabled_results) / len(tool_enabled_results)

print("Fine-tuned + tools EM:", round(tool_em, 3))
print("Fine-tuned + tools F1:", round(tool_f1, 3))


Fine-tuned + tools EM: 0.35
Fine-tuned + tools F1: 0.467


In [55]:
comparison = [
    {"Model": "Base LM", "Tool use": "No", "EM": round(base_em, 3), "F1": round(base_f1, 3)},
    {"Model": "Fine-tuned LM", "Tool use": "No", "EM": round(ft_em, 3), "F1": round(ft_f1, 3)},
    {"Model": "Fine-tuned LM", "Tool use": "Yes", "EM": round(tool_em, 3), "F1": round(tool_f1, 3)},
]

for row in comparison:
    print(row)


{'Model': 'Base LM', 'Tool use': 'No', 'EM': 0.0, 'F1': 0.019}
{'Model': 'Fine-tuned LM', 'Tool use': 'No', 'EM': 0.0, 'F1': 0.0}
{'Model': 'Fine-tuned LM', 'Tool use': 'Yes', 'EM': 0.35, 'F1': 0.467}
